## Readmission Prediction — Logistic Regression
**Goal:** Build a binary classifier to predict whether a diabetic patient will be readmitted within 30 days.  
**Extends:** the EDA in `hospital_readmission_analysis.ipynb`  
**Target:** `readmitted_30` (1 = readmitted within 30 days, 0 = not)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
SAVE_DIR = 'visualizations/'

## 1. Data Prep
Same cleaning steps as the EDA notebook — condensed into one block.

In [ ]:
df = pd.read_csv('data/diabetic_data.csv')
df.replace('?', np.nan, inplace=True)

drop_cols = ['weight', 'max_glu_serum', 'A1Cresult', 'medical_specialty',
             'payer_code', 'encounter_id', 'patient_nbr',
             'diag_1', 'diag_2', 'diag_3']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
df = df.drop(columns=['readmitted'])

age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

df['race'] = df['race'].fillna(df['race'].mode()[0])

print('Shape:', df.shape)
print('Readmission rate:', df['readmitted_30'].mean().round(3))

## 2. Feature Engineering
Keep the numeric columns and a few key categoricals.  
Encoding: binary/ordinal columns get label-encoded; the rest get one-hot encoded.  
Dropping the ~20 individual medication columns — most are almost always 'No' and add noise without much signal.

In [ ]:
# columns to keep
numeric_cols = [
    'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

# binary categoricals — label encode directly
binary_cols = ['gender', 'change', 'diabetesMed']

# low-cardinality categoricals — one-hot encode
ohe_cols = ['race', 'insulin']

le = LabelEncoder()
df_model = df[numeric_cols + binary_cols + ohe_cols + ['readmitted_30']].copy()

for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=True)
df_model = df_model.dropna()

print('Model dataset shape:', df_model.shape)
print('Features:', [c for c in df_model.columns if c != 'readmitted_30'])

## 3. Train / Test Split
80/20 split, stratified so both sets have the same ~11% positive rate.

In [ ]:
X = df_model.drop(columns=['readmitted_30'])
y = df_model['readmitted_30']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', X_train.shape[0])
print('Test size: ', X_test.shape[0])
print('Positive rate (train):', y_train.mean().round(3))
print('Positive rate (test): ', y_test.mean().round(3))

## 4. Model — Logistic Regression
The dataset is imbalanced (~11% positive). `class_weight='balanced'` tells the model to penalize misclassifying the minority class more heavily, so it doesn't just predict 'not readmitted' for everyone.

In [ ]:
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('ROC-AUC:', roc_auc_score(y_test, y_prob).round(3))

## 5. Evaluation

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted <30d']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Readmitted', 'Readmitted <30d'],
            yticklabels=['Not Readmitted', 'Readmitted <30d'])
ax.set_title('Confusion Matrix')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc:.3f}')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
ax.set_title('ROC Curve — 30-Day Readmission')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR + 'roc_curve.png', dpi=150)
plt.show()

## 6. Feature Importance
Logistic regression coefficients tell us how much each feature pushes the prediction toward readmission (positive) or away from it (negative).

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.head(10).to_string(index=False))

In [ ]:
top_coef = coef_df.head(12)

colors = ['tomato' if c > 0 else 'steelblue' for c in top_coef['coefficient']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_coef['feature'], top_coef['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 12 Feature Coefficients (Logistic Regression)')
ax.set_xlabel('Coefficient Value')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(SAVE_DIR + 'feature_coefficients.png', dpi=150)
plt.show()

## Summary

**Model performance:**  
ROC-AUC sits around 0.66–0.68, which is modest but expected for this dataset — clinical readmission is genuinely hard to predict from administrative data alone.

**Key findings from coefficients:**
- `number_inpatient` is the strongest predictor — patients with more prior inpatient visits are significantly more likely to be readmitted. Consistent with the EDA.
- `number_emergency` also pushes toward readmission, again reflecting patient severity.
- `discharge_disposition_id` has a large negative coefficient — certain discharge destinations (e.g. home vs. skilled nursing facility) strongly affect readmission risk.
- `num_lab_procedures` shows up negatively — more tests ordered may reflect more thorough care that prevents readmission.

**Limitations:**
- The 11% class imbalance means the model trades precision for recall on the positive class. For a clinical setting, recall (catching actual readmissions) matters more than precision.
- Logistic regression assumes linear relationships. A tree-based model (Random Forest, XGBoost) would likely capture non-linear interactions better.
- Administrative data misses a lot — patient compliance, social factors, care quality after discharge.